#  Player Transfer Value Prediction
### Regression — Estimating Football Player Market Value

---

> **Algorithm:** Random Forest Regressor with GridSearchCV  
> **Dataset:** 10,754 professional players with career statistics & market values  
> **Target:** `current_value` (player market value in €)

---

##  Problem Statement

Transfer fees are one of the most consequential financial decisions a football club makes. Understanding what drives market value — and being able to estimate it from objective statistics — gives clubs a quantitative edge in negotiations, avoids overpaying, and helps identify undervalued targets.

This notebook builds a **Random Forest regression model** to predict the current market value of professional football players from performance statistics, age, injury history, and team context.

###  Notebook Structure
1. Imports & Configuration
2. Dataset Overview
3. Data Cleaning & Deduplication
4. Feature Engineering (Per-90 Stats + Team Encoding)
5. Exploratory Data Analysis
6. Feature Selection via Correlation
7. Model Building & Hyperparameter Tuning
8. Model Evaluation
9. Key Insights & Conclusion

---
## 1.  Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ── Global settings 
RANDOM_STATE    = 42
CORR_THRESHOLD  = 0.05   # drop features with |correlation| < this

np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120})

print('✅ Libraries loaded.')

---
## 2.  Dataset Overview

The dataset contains **10,754 professional football players** with career statistics, physical attributes, and market valuations sourced from Transfermarkt and stats providers.

Key columns:
| Column | Description |
|--------|-------------|
| `current_value` | Current market value (€) — **target** |
| `highest_value` | Historical peak value (€) |
| `age` | Player age |
| `appearance` | Number of appearances |
| `Goals` / `Assists` | Raw season totals |
| `minutes played` | Minutes on the pitch |
| `days_injured` / `games_injured` | Injury burden |
| `award` | Individual awards won |
| `position_encoded` | Encoded position |
| `winger` | Binary flag — is the player a winger? |

In [ ]:
# ── Load data 
df_raw = pd.read_csv('/kaggle/input/transfer_value_prediction_dataset.csv')

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
display(df_raw.head())

In [ ]:
df_raw.info()

In [ ]:
display(df_raw[['age', 'Goals', 'Assists', 'minutes played',
                'current_value', 'highest_value', 'days_injured']].describe())

---
## 3.  Data Cleaning & Deduplication

The dataset contains multiple rows per player (different seasons or data sources). We **deduplicate on `player`** to retain one record per player, which is appropriate for predicting current market value (a point-in-time estimate).

In [ ]:
df = df_raw.copy()

# ── Deduplicate: keep one row per unique player 
n_before = len(df)
df = df.drop_duplicates(subset=['player'])
print(f'Rows removed (duplicates): {n_before - len(df):,}')
print(f'Unique players remaining : {len(df):,}')

# ── Missing value audit 
total_missing = df.isnull().sum().sum()
if total_missing > 0:
    print(f'\nMissing values found: {total_missing}')
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print('\n✅ No missing values.')

---
## 4.  Feature Engineering

### 4.1 Per-90-Minute Normalisation

Raw counting stats (goals, assists, cards) favour players with more appearances. Converting to **per-90-minute rates** removes this bias and enables fair comparison between a 60-min substitute and a regular starter.

In [ ]:
counting_cols = [
    'Goals', 'Assists', 'Yellow Cards',
    'Second Yellow Card', 'Red Card',
    'Goal Conceded', 'Clean Sheets'
]

# Guard against division by zero
df['minutes played'] = df['minutes played'].replace(0, np.nan)

for col in counting_cols:
    df[f'{col}_per90'] = (df[col] * 90) / df['minutes played']

df['minutes played'] = df['minutes played'].fillna(0)

print('Per-90 features created:')
for col in counting_cols:
    print(f'  {col} → {col}_per90')

### 4.2 Team Target Encoding

A player's team is a strong proxy for their value — elite clubs pay premiums. We encode team by the **mean current value of all players at that club**. This captures the team prestige effect without creating hundreds of dummy variables.

In [ ]:
team_mean_value = df.groupby('team')['current_value'].mean()
df['team_encoded'] = df['team'].map(team_mean_value)

# Save encoding for inference
joblib.dump(team_mean_value, 'team_encoding.pkl')
print(' Team encoding saved → team_encoding.pkl')

# Preview top-valued teams
print('\nTop 10 teams by mean player value:')
display(team_mean_value.sort_values(ascending=False).head(10)
        .apply(lambda x: f'€{x/1e6:.1f}M').rename('Mean Player Value'))

In [ ]:
# ── Drop identifier and raw (now replaced) columns 
model_df = df.drop(columns=['name', 'player', 'team', 'position'] + counting_cols)
print(f'Modelling dataset shape: {model_df.shape}')

---
## 5. 📊 Exploratory Data Analysis

### 5.1 Transfer Value Distribution

Market values are extremely right-skewed — a handful of elite players have values that dwarf the median.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw distribution
model_df['current_value'].plot(kind='hist', bins=60, ax=axes[0],
                                color='#1565c0', edgecolor='white')
axes[0].set_title('Transfer Value — Raw Distribution')
axes[0].set_xlabel('Current Value (€)')

# Log-transformed
np.log1p(model_df['current_value']).plot(kind='hist', bins=60, ax=axes[1],
                                           color='#2e7d32', edgecolor='white')
axes[1].set_title('Transfer Value — Log-Transformed')
axes[1].set_xlabel('log(Current Value + 1)')

plt.suptitle('Market Value Distribution (10,754 Players)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Median value : €{model_df["current_value"].median():,.0f}')
print(f'Mean value   : €{model_df["current_value"].mean():,.0f}')
print(f'Max value    : €{model_df["current_value"].max():,.0f}')
print(f'% worth < €1M: {(model_df["current_value"] < 1e6).mean():.1%}')

### 5.2 Value vs Age — The Career Value Arc

In [ ]:
age_value = df.groupby('age')['current_value'].median() / 1e6  # median in €M

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(age_value.index, age_value.values, color='#d32f2f', alpha=0.8, edgecolor='white')
ax.set_xlabel('Age')
ax.set_ylabel('Median Current Value (€M)')
ax.set_title('Median Transfer Value by Age\n(the career value arc)', fontsize=13)
plt.tight_layout()
plt.show()

### 5.3 Current Value vs Highest Historical Value

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sample = df.sample(min(2000, len(df)), random_state=RANDOM_STATE)
ax.scatter(sample['highest_value'] / 1e6, sample['current_value'] / 1e6,
           alpha=0.3, color='#4a148c', edgecolors='white', linewidths=0.3, s=20)
ax.set_xlabel('Highest Historical Value (€M)')
ax.set_ylabel('Current Value (€M)')
ax.set_title('Peak Value vs Current Value\n(players above diagonal regained their peak)', fontsize=12)
lim = max(sample['highest_value'].max(), sample['current_value'].max()) / 1e6
ax.plot([0, lim], [0, lim], 'r--', linewidth=1.2, label='Current = Peak')
ax.legend()
plt.tight_layout()
plt.show()

corr_val = df[['highest_value', 'current_value']].corr().iloc[0, 1]
print(f'Pearson correlation (highest_value vs current_value): {corr_val:.4f}')

---
## 6.  Feature Selection via Correlation

We standardise features first (so correlations are on a level playing field) then remove any feature with an absolute correlation below `{CORR_THRESHOLD}` with the target.

In [ ]:
features = model_df.drop(columns=['current_value'])
target   = model_df['current_value']

# Standardise for fair correlation analysis
scaler = StandardScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)
analysis_df     = pd.concat([features_scaled, target.reset_index(drop=True)], axis=1)

corr_with_target = analysis_df.corr()['current_value'].sort_values(key=abs, ascending=False)

# Heatmap
corr_abs = corr_with_target.abs().drop('current_value').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 9))
sns.heatmap(corr_abs.to_frame(), annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.6})
ax.set_title('|Correlation| with Current Value', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Identify and drop weak features
weak_features = corr_with_target[
    (corr_with_target.abs() < CORR_THRESHOLD) & (corr_with_target.index != 'current_value')
].index.tolist()

print(f'Dropping {len(weak_features)} features with |r| < {CORR_THRESHOLD}:')
for f in weak_features:
    print(f'  - {f}  (|r| = {corr_with_target[f]:.4f})')

selected_features = corr_with_target[
    (corr_with_target.abs() >= CORR_THRESHOLD) & (corr_with_target.index != 'current_value')
].index.tolist()

print(f'\n Selected {len(selected_features)} features for modelling.')

---
## 7. 🤖 Model Building & Hyperparameter Tuning

### 7.1 Train / Test Split

In [ ]:
final_df = model_df.drop(columns=weak_features, errors='ignore')

X = final_df.drop(columns=['current_value'])
y = final_df['current_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set : {len(X_train):,} samples')
print(f'Test set     : {len(X_test):,}  samples')
print(f'Features     : {X.shape[1]}')

### 7.2 Pipeline + GridSearchCV

| Hyperparameter | Values Tried | Effect |
|---|---|---|
| `n_estimators` | 150, 200 | More trees = lower variance, more compute |
| `max_depth` | 6, 8 | Controls tree depth — prevents overfitting |

**Scoring metric:** R² — proportion of variance explained by the model.

In [ ]:
pipeline = Pipeline([('model', RandomForestRegressor(random_state=RANDOM_STATE))])

param_grid = {
    'model__n_estimators': [150, 200],
    'model__max_depth':    [6, 8]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print(f'\nBest Hyperparameters : {grid.best_params_}')
print(f'Best CV R²           : {grid.best_score_:.4f}')

In [ ]:
# GridSearch CV results table
cv_df = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
display(cv_df[['param_model__n_estimators', 'param_model__max_depth',
               'mean_test_score', 'std_test_score', 'rank_test_score']]
        .rename(columns={
            'param_model__n_estimators': 'n_estimators',
            'param_model__max_depth': 'max_depth',
            'mean_test_score': 'mean_R²',
            'std_test_score': 'std_R²',
            'rank_test_score': 'rank'
        }).reset_index(drop=True))

---
## 8.  Model Evaluation

### 8.1 R² Score

In [ ]:
best_model   = grid.best_estimator_
y_pred_test  = best_model.predict(X_test)
y_pred_train = best_model.predict(X_train)

r2_test  = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f'Train R²    : {r2_train:.4f}')
print(f'Test  R²    : {r2_test:.4f}')
print(f'Test  MAE   : €{mae_test:,.0f}')
print(f'Test  RMSE  : €{rmse_test:,.0f}')

### 8.2 Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual (in €M)
axes[0].scatter(y_test / 1e6, y_pred_test / 1e6,
                alpha=0.3, color='#4a148c', edgecolors='white', linewidths=0.3, s=18)
lim = max(y_test.max(), y_pred_test.max()) / 1e6
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Value (€M)')
axes[0].set_ylabel('Predicted Value (€M)')
axes[0].set_title(f'Actual vs Predicted Transfer Value\nR² = {r2_test:.3f}')
axes[0].legend(fontsize=9)

# Feature importance (top 12)
feat_imp = pd.Series(
    best_model.named_steps['model'].feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(12)

feat_imp.plot(kind='barh', ax=axes[1], color='#4a148c', edgecolor='white')
axes[1].invert_yaxis()
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('Top-12 Feature Importances\n(transfer value drivers)')

plt.suptitle('Transfer Value Predictor — Model Results', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 8.3 Residual Analysis

In [ ]:
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residual distribution
axes[0].hist(residuals / 1e6, bins=50, color='#d32f2f', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='black', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Residual (€M)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distribution')

# Residuals vs Predicted
axes[1].scatter(y_pred_test / 1e6, residuals / 1e6,
                alpha=0.3, color='#1565c0', edgecolors='white', linewidths=0.3, s=18)
axes[1].axhline(y=0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Value (€M)')
axes[1].set_ylabel('Residual (€M)')
axes[1].set_title('Residuals vs Predicted Values')

plt.suptitle('Residual Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Mean residual  : €{residuals.mean():,.0f}  (should be ≈ 0)')
print(f'Std of residuals: €{residuals.std():,.0f}')

### 8.4 Highest-Value Players — Predicted vs Actual

In [ ]:
results_df = X_test.copy()
results_df['actual_value']    = y_test.values
results_df['predicted_value'] = y_pred_test
results_df['error_pct']       = (
    abs(results_df['predicted_value'] - results_df['actual_value'])
    / results_df['actual_value'].replace(0, np.nan) * 100
)

# Top 10 most valuable players in the test set
top_players = results_df.nlargest(10, 'actual_value')[['actual_value', 'predicted_value', 'error_pct']]
top_players['actual_value']    = top_players['actual_value'].apply(lambda x: f'€{x/1e6:.1f}M')
top_players['predicted_value'] = top_players['predicted_value'].apply(lambda x: f'€{x/1e6:.1f}M')
top_players['error_pct']       = top_players['error_pct'].apply(lambda x: f'{x:.1f}%')

print('Top 10 Highest-Value Players in Test Set:')
display(top_players)

---
## 9.  Save Model & Output

In [ ]:
# Save trained model
joblib.dump(best_model, 'transfer_value_prediction_model.pkl', compress=3)
print('✅ Model saved → transfer_value_prediction_model.pkl')

# Save processed dataset
final_df.to_csv('transfer_value_prediction_processed_dataset.csv', index=False)
print('✅ Dataset saved → transfer_value_prediction_processed_dataset.csv')

---
## 10.  Key Insights & Conclusion

### What drives player market value?

| Driver | Insight |
|--------|---------|
| **Peak historical value** | The single strongest predictor — clubs pay a premium for proven pedigree |
| **Age** | Values peak around age 24–26 and decline sharply after 30; the market discounts aging |
| **Team context** | Players at elite clubs (high `team_encoded`) carry a prestige premium |
| **Goals & Assists per 90** | Per-90 normalisation is more informative than raw tallies — efficiency matters |
| **Injury burden** | `days_injured` negatively correlates with value — injury-prone players are discounted |
| **Awards** | Individual accolades (Ballon d'Or nominations, league Best XI) significantly boost market price |

### Why Random Forest works well here:
- Transfer value is a **non-linear function** of many interacting factors — tree ensembles handle this naturally
- Robust to outliers (€180M player doesn't break the model)
- Built-in feature importance for interpretability

### Model limitations:
- **Extreme values are underestimated** — the model regresses to the mean for elite players (seen in residual plot)
- **No contract data** — years remaining and release clause are major real-world value drivers
- **Snapshot model** — value is dynamic; retraining annually is essential

### Future improvements:
- Add contract data (years remaining, release clause amount)
- Incorporate social media metrics (following, brand value)
- Use XGBoost with log-transformed target to handle the right-skewed distribution
- Build a position-specific model (GKs valued differently from strikers)
- Deploy via REST API for real-time valuation in a scouting dashboard